# Tuần 6 - Phần 1: Stress Test với Dữ liệu Hỗn loạn (Adversarial DGP / Data Shift)

Một mô hình Data Science chỉ thực sự có giá trị khi nó sống sót được qua các bài kiểm tra khắc nghiệt (Stress Test). 
Giả sử tháng tới, thị trường thay đổi: Khách hàng bị **"lờn khuyến mãi" (Treatment Effect Decay)**, tức là họ nhận mã giảm giá quá nhiều nên không còn hào hứng đi thêm chuyến nữa.

Trong Notebook này, chúng ta sẽ giả lập kịch bản **Giảm 60% tác dụng của Voucher**. Hãy xem điều gì sẽ xảy ra với các chiến lược Marketing cũ, và thuật toán AI Profit Targeting sẽ cứu công ty bàn thua trông thấy như thế nào.

In [1]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split
import os

base_path = r"D:\Intern VSF\GSM-promotion-experimentation"
data_path = os.path.join(base_path, 'data', 'processed', 'segmented_simulation_data.csv')
df = pd.read_csv(data_path)
df['is_credit_card'] = (df['payment_type'] == 1).astype(int)

print("============================================================")
print("Feature 2: Adversarial DGP (Treatment Effect Decay)")
print("============================================================")

Feature 2: Adversarial DGP (Treatment Effect Decay)


## 1. Giả lập kịch bản "Lờn khuyến mãi" (DGP Shift)

In [2]:
# Khách hàng lờn khuyến mãi: Tác động thật (cate_true) giảm 60%
df['cate_true_decayed'] = df['cate_true'] * 0.4

# Sinh lại số chuyến đi quan sát được (Y_rand) dựa trên ITE mới
np.random.seed(42)
# Base trips (Y0) remains the same. Y0 = Y_rand if T=0 else Y_rand - cate_true
df['Y0_implied'] = np.where(df['treatment_rand'] == 0, df['Y_rand'], df['Y_rand'] - df['cate_true'])

# New Y_rand_decayed = Y0 + T * cate_true_decayed + noise
noise = np.random.normal(0, 0.5, size=len(df))
df['Y_rand_decayed'] = df['Y0_implied'] + df['treatment_rand'] * df['cate_true_decayed'] + noise
# Đảm bảo số chuyến không bị âm
df['Y_rand_decayed'] = np.maximum(0, np.round(df['Y_rand_decayed']))

print(f"CATE trung bình ban đầu: {df['cate_true'].mean():.2f} chuyến")
print(f"CATE trung bình sau khi bị lờn (Decay): {df['cate_true_decayed'].mean():.2f} chuyến")

CATE trung bình ban đầu: 0.80 chuyến
CATE trung bình sau khi bị lờn (Decay): 0.32 chuyến


## 2. Auxiliary X-Learner Stress Model (không phải champion hiện tại)
Cell này được giữ như một sensitivity experiment lịch sử. Champion dùng trong policy pipeline là simplified R-Learner-style residual model.

In [3]:
features = ['age', 'is_urban', 'preferred_hour', 'is_rush_hour', 'is_airport_trip',
            'is_rain_rider', 'is_weekend_rider', 'is_credit_card', 'passenger_count',
            'monthly_rides_history', 'recency_days']

X = df[features]
y = df['Y_rand_decayed']
T = df['treatment_rand']

X_tv, X_test, y_tv, y_test, T_tv, T_test = train_test_split(X, y, T, test_size=0.2, random_state=42)
X_train, X_val, y_train, y_val, T_train, T_val = train_test_split(X_tv, y_tv, T_tv, test_size=0.25, random_state=42)

params = dict(random_state=42, min_child_weight=5, reg_lambda=1.0, n_estimators=200, learning_rate=0.05, max_depth=4)

m0 = xgb.XGBRegressor(**params)
m1 = xgb.XGBRegressor(**params)
m0.fit(X_train[T_train == 0], y_train[T_train == 0])
m1.fit(X_train[T_train == 1], y_train[T_train == 1])

pseudo0 = m1.predict(X_train[T_train == 0]) - y_train[T_train == 0]
pseudo1 = y_train[T_train == 1] - m0.predict(X_train[T_train == 1])

tau0 = xgb.XGBRegressor(**params); tau0.fit(X_train[T_train == 0], pseudo0)
tau1 = xgb.XGBRegressor(**params); tau1.fit(X_train[T_train == 1], pseudo1)

cate_decayed = 0.5 * tau0.predict(X_test) + 0.5 * tau1.predict(X_test)
pred1_decayed = m1.predict(X_test)
print(f"AI X-Learner dự đoán CATE trung bình (Decayed): {cate_decayed.mean():.4f}")

AI X-Learner dự đoán CATE trung bình (Decayed): 0.5741


## 3. Đánh giá Lợi nhuận: AI vs Marketing Truyền thống

In [4]:
df_test = df.loc[X_test.index].copy()
df_test['avg_fare'] = df_test['avg_fare_per_trip']

VOUCHER_RATE = 0.15
MARGIN_RATE  = 0.70

df_test['cate_pred_decayed'] = cate_decayed
df_test['pred_rides_treated'] = pred1_decayed
df_test['voucher_cost'] = df_test['avg_fare'] * VOUCHER_RATE
df_test['margin_per_ride'] = df_test['avg_fare'] * MARGIN_RATE

df_test['expected_value'] = (df_test['cate_pred_decayed'] * df_test['margin_per_ride']) - (df_test['pred_rides_treated'] * df_test['voucher_cost'])

def evaluate_policy_decayed(target_mask, df_eval, label):
    targeted = df_eval[target_mask]
    n_targeted = target_mask.sum()
    if n_targeted == 0:
        return {"Policy": label, "N_Targeted": 0, "Total_Voucher_Cost": 0, "Expected_Incremental_Profit": 0}
    total_ev = targeted['expected_value'].sum()
    total_voucher_cost = (targeted['pred_rides_treated'] * targeted['voucher_cost']).sum()
    return {
        "Policy": label,
        "N_Targeted": int(n_targeted),
        "Total_Voucher_Cost": round(total_voucher_cost, 0),
        "Expected_Incremental_Profit": round(total_ev, 0)
    }

results = []
mass_mask = pd.Series([True] * len(df_test), index=df_test.index)
results.append(evaluate_policy_decayed(mass_mask, df_test, "1. Mass Voucher (Marketing truyền thống)"))

suburban_mask = df_test['persona'].str.contains('Suburban', case=False, na=False)
results.append(evaluate_policy_decayed(suburban_mask, df_test, "2. Segment Targeting (quy tắc Suburban)"))

profit_mask = df_test['expected_value'] > 0
results.append(evaluate_policy_decayed(profit_mask, df_test, "3. AI Profit Targeting (Tự động thích nghi)"))

display(pd.DataFrame(results))

,Policy,N_Targeted,Total_Voucher_Cost,Expected_Incremental_Profit
0,1. Mass Voucher (Marketing truyền thống),4000,126101.0,-83025.0
1,2. Segment Targeting (quy tắc Suburban),1660,31556.0,-16344.0
2,3. AI Profit Targeting (Tự động thích nghi),521,9236.0,5448.0


### Kết luận (Business Implication):
Khi thị trường thay đổi (khách hàng có dấu hiệu lờn khuyến mãi):
- Nếu sử dụng quy tắc **Phát đại trà (Mass Voucher)** hoặc **Phát theo tệp cố định (Segment)**, chiến dịch sẽ gặp rủi ro tài chính cao do tiếp tục phân bổ voucher cho tập khách hàng không còn phản ứng tích cực.
- Ngược lại, phương pháp **AI Profit Targeting** thể hiện khả năng tự động thích nghi. Khi giá trị kỳ vọng (Expected Value) giảm, mô hình tự động thu hẹp quy mô mục tiêu (N_Targeted giảm mạnh), giúp bảo vệ ngân sách và hạn chế rủi ro đốt tiền.